In [1]:
import pandas as pd

df = pd.read_csv("../data/nasa_glc.csv")
df.shape

(11033, 31)

In [2]:
trig = df["landslide_trigger"].fillna("").astype(str).str.lower()
pattern = r"rain|storm|monsoon|hurricane|cyclone|typhoon|downpour"
df["y"] = trig.str.contains(pattern, regex=True).astype(int)

df["y"].value_counts(normalize=True)

y
1    0.78945
0    0.21055
Name: proportion, dtype: float64

In [3]:
import numpy as np
import pandas as pd

use = df.dropna(subset=["latitude", "longitude", "y"]).copy()

# Ensure numeric
use["latitude"] = pd.to_numeric(use["latitude"], errors="coerce")
use["longitude"] = pd.to_numeric(use["longitude"], errors="coerce")
use = use.dropna(subset=["latitude", "longitude"])

X = pd.DataFrame({
    "lat": use["latitude"],
    "lon": use["longitude"],
})
X["lat_sin"] = np.sin(np.radians(X["lat"]))
X["lat_cos"] = np.cos(np.radians(X["lat"]))
X["lon_sin"] = np.sin(np.radians(X["lon"]))
X["lon_cos"] = np.cos(np.radians(X["lon"]))

y = use["y"].astype(int).values

X.shape, y.shape

((11033, 6), (11033,))

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=400,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)

probs = model.predict_proba(X_test)[:, 1]
preds = (probs >= 0.5).astype(int)

print("AUC:", roc_auc_score(y_test, probs))
print(classification_report(y_test, preds, digits=3))

AUC: 0.8032258064516129
              precision    recall  f1-score   support

           0      0.761     0.417     0.539       465
           1      0.861     0.965     0.910      1742

    accuracy                          0.850      2207
   macro avg      0.811     0.691     0.725      2207
weighted avg      0.840     0.850     0.832      2207



In [5]:
import joblib
from pathlib import Path

Path("../models").mkdir(exist_ok=True)

artifact = {
    "model": model,
    "features": list(X.columns),
    "version": "v1_latlon_keywords",
    "rain_words": ["rain", "storm", "monsoon", "hurricane", "cyclone", "typhoon", "downpour"],
}

joblib.dump(artifact, "../models/risk_model.joblib")
"saved ../models/risk_model.joblib"

'saved ../models/risk_model.joblib'

In [6]:
def featurize(lat, lon):
    lat = float(lat); lon = float(lon)
    row = {
        "lat": lat,
        "lon": lon,
        "lat_sin": np.sin(np.radians(lat)),
        "lat_cos": np.cos(np.radians(lat)),
        "lon_sin": np.sin(np.radians(lon)),
        "lon_cos": np.cos(np.radians(lon)),
    }
    return pd.DataFrame([row])[artifact["features"]]

print(float(model.predict_proba(featurize(22.57, 88.36))[0, 1]))
print(float(model.predict_proba(featurize(37.77, -122.42))[0, 1]))

0.8049472860837998
0.9581096854534354
